In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
import gradio as gr
import re
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# Global Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = './data/'
MODEL_PATH = './data/zsl_f1_model.pth'

print(f"Using device: {DEVICE}")

def clean_text(text):
    """Clean text data"""
    if pd.isna(text):
        return 'Unknown'
    return re.sub(r'\s+', ' ', str(text)).strip()

def load_f1_data():
    """Load and clean F1 datasets"""
    print("Loading F1 data...")
    
    # Load datasets
    winners = pd.read_csv(DATA_DIR + 'winners.csv')
    drivers = pd.read_csv(DATA_DIR + 'drivers_updated.csv') 
    teams = pd.read_csv(DATA_DIR + 'teams_updated.csv')
    fastest_laps = pd.read_csv(DATA_DIR + 'fastest_laps_updated.csv')
    
    # Clean text columns
    for df in [winners, drivers, teams, fastest_laps]:
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].apply(clean_text)
    
    # Process dates and years
    winners['year'] = pd.to_datetime(winners['Date']).dt.year
    drivers['year'] = drivers['year'].astype(int)
    teams['year'] = teams['year'].astype(int)
    fastest_laps['year'] = fastest_laps['year'].astype(int)
    
    # Rename for consistency
    drivers = drivers.rename(columns={'Car': 'Team'})
    
    # Convert positions to numeric
    drivers['Pos'] = pd.to_numeric(drivers['Pos'], errors='coerce')
    
    print(f"Loaded {len(winners)} races, {len(drivers)} driver entries")
    
    return winners, drivers, teams, fastest_laps

def create_driver_features(drivers, teams, fastest_laps):
    """Create features for drivers"""
    print("Creating driver features...")
    
    # Sort by driver and year
    drivers = drivers.sort_values(['Driver', 'year']).reset_index(drop=True)
    
    # Previous year performance
    drivers['prev_year_pts'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['prev_year_pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(20)
    
    # Experience (years in F1)
    drivers['experience'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    
    # Team features
    teams = teams.sort_values(['Team', 'year']).reset_index(drop=True)
    teams['prev_team_pts'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['prev_team_pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(10)
    
    # Merge team features
    drivers = drivers.merge(
        teams[['Team', 'year', 'prev_team_pts', 'prev_team_pos']], 
        on=['Team', 'year'], 
        how='left'
    ).fillna(0)
    
    # Fastest laps count
    fl_count = fastest_laps.groupby(['Driver', 'year']).size().reset_index(name='fastest_laps')
    drivers = drivers.merge(fl_count, on=['Driver', 'year'], how='left').fillna(0)
    
    # Fill any remaining NaN values
    numeric_cols = ['prev_year_pts', 'prev_year_pos', 'experience', 'prev_team_pts', 'prev_team_pos', 'fastest_laps']
    for col in numeric_cols:
        drivers[col] = drivers[col].fillna(0)
    
    print(f"Created features for {len(drivers)} driver entries")
    
    return drivers

def get_home_country(gp_name):
    """Get country code from Grand Prix name"""
    country_map = {
        'British': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'German': 'GER',
        'Belgian': 'BEL', 'French': 'FRA', 'Dutch': 'NED', 'Spanish': 'ESP',
        'Brazilian': 'BRA', 'Japanese': 'JPN', 'Canadian': 'CAN', 'Austrian': 'AUT',
        'Hungarian': 'HUN', 'Mexican': 'MEX', 'Australian': 'AUS', 'United States': 'USA'
    }
    
    for key, country in country_map.items():
        if key in str(gp_name):
            return country
    return None

def build_race_dataset(winners, drivers_with_features):
    """Build dataset for modeling"""
    print("Building race dataset...")
    
    data = []
    
    for _, race in winners.iterrows():
        year = race['year'] 
        gp = race['Grand Prix']
        winner = race['Winner']
        
        # Get drivers for this year
        year_drivers = drivers_with_features[drivers_with_features['year'] == year]
        
        if len(year_drivers) == 0:
            continue
            
        race_country = get_home_country(gp)
        
        for _, driver in year_drivers.iterrows():
            # Check if this is a home race
            is_home = 1 if race_country and driver['Nationality'] == race_country else 0
            
            record = {
                'year': year,
                'grand_prix': gp,
                'driver': driver['Driver'],
                'team': driver['Team'], 
                'nationality': driver['Nationality'],
                'prev_year_pts': driver['prev_year_pts'],
                'prev_year_pos': driver['prev_year_pos'],
                'experience': driver['experience'],
                'prev_team_pts': driver['prev_team_pts'],
                'prev_team_pos': driver['prev_team_pos'],
                'fastest_laps': driver['fastest_laps'],
                'is_home': is_home,
                'is_winner': 1 if driver['Driver'] == winner else 0
            }
            
            data.append(record)
    
    df = pd.DataFrame(data)
    print(f"Built dataset with {len(df)} records, {df['is_winner'].sum()} winners")
    
    return df

class SemanticEncoder:
    """Encode categorical features using TF-IDF"""
    
    def __init__(self, max_features=50):
        self.max_features = max_features
        self.encoders = {}
        self.fitted = False
        
    def fit_transform(self, categories):
        """Fit and transform categorical data"""
        
        all_texts = []
        ranges = {}
        start = 0
        
        for cat_name, values in categories.items():
            texts = []
            for val in values:
                if cat_name == 'driver':
                    text = f"{val} formula one racing driver"
                elif cat_name == 'team':
                    text = f"{val} formula one team constructor"
                elif cat_name == 'grand_prix':
                    text = f"{val} grand prix race circuit"
                elif cat_name == 'nationality':
                    text = f"{val} country nation"
                else:
                    text = str(val)
                texts.append(text)
            
            all_texts.extend(texts)
            ranges[cat_name] = (start, start + len(texts))
            start += len(texts)
        
        # Fit TF-IDF
        vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            stop_words='english',
            lowercase=True
        )
        
        features = vectorizer.fit_transform(all_texts).toarray()
        
        # Split features by category
        result = {}
        for cat_name, (start_idx, end_idx) in ranges.items():
            result[cat_name] = features[start_idx:end_idx]
            
        self.fitted = True
        self.vectorizer = vectorizer
        
        return result

def prepare_data(df):
    """Prepare data for training"""
    print("Preparing data...")
    
    # Define columns
    cat_cols = ['grand_prix', 'driver', 'team', 'nationality']
    num_cols = ['year', 'prev_year_pts', 'prev_year_pos', 'experience', 
                'prev_team_pts', 'prev_team_pos', 'fastest_laps', 'is_home']
    
    # Split by race (to avoid data leakage)
    df['race_id'] = df['year'].astype(str) + '_' + df['grand_prix']
    unique_races = df['race_id'].unique()
    
    train_races, test_races = train_test_split(unique_races, test_size=0.2, random_state=42)
    
    train_df = df[df['race_id'].isin(train_races)].copy()
    test_df = df[df['race_id'].isin(test_races)].copy()
    
    # Remove race_id
    train_df = train_df.drop('race_id', axis=1)
    test_df = test_df.drop('race_id', axis=1)
    
    # Encode categorical features
    label_encoders = {}
    cat_dims = {}
    
    for col in cat_cols:
        le = LabelEncoder()
        
        # Fit on combined data to ensure consistency
        all_values = pd.concat([train_df[col], test_df[col]]).astype(str)
        le.fit(all_values)
        
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))
        
        label_encoders[col] = le
        cat_dims[col] = len(le.classes_)
    
    # Create semantic features
    categories = {}
    for col in cat_cols:
        categories[col] = label_encoders[col].classes_
        
    semantic_encoder = SemanticEncoder()
    semantic_features = semantic_encoder.fit_transform(categories)
    
    # Scale numerical features
    scaler = StandardScaler()
    train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
    test_df[num_cols] = scaler.transform(test_df[num_cols])
    
    print(f"Train: {len(train_df)} samples, Test: {len(test_df)} samples")
    
    return (train_df, test_df, cat_cols, num_cols, 
            label_encoders, scaler, semantic_features, cat_dims)

class F1Dataset(Dataset):
    """PyTorch dataset for F1 data"""
    
    def __init__(self, df, cat_cols, num_cols):
        self.cat_data = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num_data = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.targets = torch.tensor(df['is_winner'].values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.targets)
        
    def __getitem__(self, idx):
        return self.cat_data[idx], self.num_data[idx], self.targets[idx]

class ZeroShotF1Model(nn.Module):
    """Zero-shot F1 prediction model using semantic features"""
    
    def __init__(self, cat_dims, num_features, semantic_dim=50, embed_dim=32, hidden_dim=128):
        super().__init__()
        
        self.semantic_dim = semantic_dim
        self.embed_dim = embed_dim
        
        # Semantic projection layers
        self.gp_proj = nn.Linear(semantic_dim, embed_dim)
        self.driver_proj = nn.Linear(semantic_dim, embed_dim) 
        self.team_proj = nn.Linear(semantic_dim, embed_dim)
        self.nationality_proj = nn.Linear(semantic_dim, embed_dim)
        
        # Batch normalization for numerical features
        self.num_bn = nn.BatchNorm1d(num_features)
        
        # Main network
        total_features = 4 * embed_dim + num_features
        
        self.network = nn.Sequential(
            nn.Linear(total_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Storage for semantic features
        self.semantic_features = {}
        
    def set_semantic_features(self, semantic_features):
        """Set the semantic feature matrices"""
        device = next(self.parameters()).device
        
        for key, features in semantic_features.items():
            self.semantic_features[key] = torch.tensor(features, dtype=torch.float32).to(device)
    
    def forward(self, cat_inputs, num_inputs):
        device = cat_inputs.device
        batch_size = cat_inputs.size(0)
        
        # Ensure semantic features are on correct device
        for key in self.semantic_features:
            if self.semantic_features[key].device != device:
                self.semantic_features[key] = self.semantic_features[key].to(device)
        
        # Get semantic embeddings with bounds checking
        gp_idx = torch.clamp(cat_inputs[:, 0], 0, self.semantic_features['grand_prix'].size(0) - 1)
        driver_idx = torch.clamp(cat_inputs[:, 1], 0, self.semantic_features['driver'].size(0) - 1)
        team_idx = torch.clamp(cat_inputs[:, 2], 0, self.semantic_features['team'].size(0) - 1)
        nat_idx = torch.clamp(cat_inputs[:, 3], 0, self.semantic_features['nationality'].size(0) - 1)
        
        gp_semantic = self.semantic_features['grand_prix'][gp_idx]
        driver_semantic = self.semantic_features['driver'][driver_idx]
        team_semantic = self.semantic_features['team'][team_idx]
        nat_semantic = self.semantic_features['nationality'][nat_idx]
        
        # Project to embedding space
        gp_embed = self.gp_proj(gp_semantic)
        driver_embed = self.driver_proj(driver_semantic)
        team_embed = self.team_proj(team_semantic)
        nat_embed = self.nationality_proj(nat_semantic)
        
        # Combine categorical embeddings
        cat_combined = torch.cat([gp_embed, driver_embed, team_embed, nat_embed], dim=1)
        
        # Process numerical features
        num_processed = self.num_bn(num_inputs)
        
        # Combine all features
        combined = torch.cat([cat_combined, num_processed], dim=1)
        
        # Forward through network
        output = self.network(combined)
        
        return output

def create_data_loaders(train_df, test_df, cat_cols, num_cols, batch_size=32):
    """Create PyTorch data loaders"""
    
    train_dataset = F1Dataset(train_df, cat_cols, num_cols)
    test_dataset = F1Dataset(test_df, cat_cols, num_cols)
    
    # Handle class imbalance with weighted sampling
    winner_count = train_df['is_winner'].sum()
    total_count = len(train_df)
    
    weights = []
    for target in train_df['is_winner']:
        if target == 1:
            weights.append(total_count / (2 * winner_count))
        else:
            weights.append(total_count / (2 * (total_count - winner_count)))
    
    sampler = WeightedRandomSampler(weights, len(weights))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

def train_model(model, train_loader, test_loader, num_epochs=30):
    """Train the zero-shot model"""
    print("Training model...")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    best_loss = float('inf')
    patience = 5
    patience_count = 0
    
    train_losses = []
    test_losses = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        train_batches = 0
        
        for cat_batch, num_batch, targets in train_loader:
            cat_batch = cat_batch.to(DEVICE)
            num_batch = num_batch.to(DEVICE) 
            targets = targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(cat_batch, num_batch)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_batches += 1
        
        # Validation
        model.eval()
        test_loss = 0
        test_batches = 0
        
        with torch.no_grad():
            for cat_batch, num_batch, targets in test_loader:
                cat_batch = cat_batch.to(DEVICE)
                num_batch = num_batch.to(DEVICE)
                targets = targets.to(DEVICE)
                
                outputs = model(cat_batch, num_batch)
                loss = criterion(outputs.squeeze(), targets)
                
                test_loss += loss.item()
                test_batches += 1
        
        avg_train_loss = train_loss / train_batches if train_batches > 0 else 0
        avg_test_loss = test_loss / test_batches if test_batches > 0 else 0
        
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")
        
        # Early stopping
        if avg_test_loss < best_loss:
            best_loss = avg_test_loss
            patience_count = 0
            # Save best model
            torch.save({
                'model_state_dict': model.state_dict(),
                'semantic_features': model.semantic_features
            }, MODEL_PATH)
        else:
            patience_count += 1
            if patience_count >= patience:
                print("Early stopping triggered")
                break
    
    # Plot training curves
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.grid(True)
    plt.savefig('./data/training_curve.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("Training completed!")

def evaluate_model(model, test_loader):
    """Evaluate model performance"""
    print("Evaluating model...")
    
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        for cat_batch, num_batch, targets in test_loader:
            cat_batch = cat_batch.to(DEVICE)
            num_batch = num_batch.to(DEVICE)
            
            outputs = model(cat_batch, num_batch)
            probs = torch.sigmoid(outputs.squeeze())
            preds = (probs > 0.5).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_preds)
    recall = recall_score(all_targets, all_preds, zero_division=0)
    auc = roc_auc_score(all_targets, all_probs)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"AUC: {auc:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, target_names=['Not Winner', 'Winner']))
    
    return accuracy, recall, auc

class F1Predictor:
    """F1 race winner predictor"""
    
    def __init__(self):
        self.model = None
        self.label_encoders = None
        self.scaler = None
        self.cat_cols = None
        self.num_cols = None
        self.driver_data = {}
        self.years = []
        self.races = []
        
    def load_model(self, model_path, label_encoders, scaler, cat_cols, num_cols, semantic_features, cat_dims):
        """Load trained model and preprocessing objects"""
        
        # Initialize model
        self.model = ZeroShotF1Model(cat_dims, len(num_cols))
        self.model.to(DEVICE)
        
        # Load model weights
        if os.path.exists(model_path):
            checkpoint = torch.load(model_path, map_location=DEVICE)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            
        # Set semantic features
        self.model.set_semantic_features(semantic_features)
        
        self.label_encoders = label_encoders
        self.scaler = scaler
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        
    def setup_data(self, drivers_data, winners_data):
        """Setup prediction data"""
        
        # Group drivers by year
        for year, group in drivers_data.groupby('year'):
            self.driver_data[year] = group.to_dict('records')
            
        self.years = sorted(drivers_data['year'].unique())
        self.races = sorted(winners_data['Grand Prix'].unique())
        
    def predict_race(self, year, grand_prix):
        """Predict race winners"""
        
        if year not in self.driver_data:
            return "No data available for this year"
            
        self.model.eval()
        predictions = []
        
        race_country = get_home_country(grand_prix)
        
        for driver_info in self.driver_data[year]:
            
            # Prepare categorical features
            cat_features = []
            for col in self.cat_cols:
                if col == 'grand_prix':
                    value = grand_prix
                else:
                    col_map = {'driver': 'Driver', 'team': 'Team', 'nationality': 'Nationality'}
                    value = driver_info.get(col_map.get(col, col), 'Unknown')
                
                # Encode
                if str(value) in self.label_encoders[col].classes_:
                    encoded = self.label_encoders[col].transform([str(value)])[0]
                else:
                    encoded = len(self.label_encoders[col].classes_) - 1  # Unknown category
                    
                cat_features.append(encoded)
            
            # Prepare numerical features  
            num_features = []
            for col in self.num_cols:
                if col == 'is_home':
                    value = 1.0 if race_country and driver_info.get('Nationality') == race_country else 0.0
                else:
                    col_map = {
                        'year': 'year',
                        'prev_year_pts': 'prev_year_pts', 
                        'prev_year_pos': 'prev_year_pos',
                        'experience': 'experience',
                        'prev_team_pts': 'prev_team_pts',
                        'prev_team_pos': 'prev_team_pos', 
                        'fastest_laps': 'fastest_laps'
                    }
                    value = float(driver_info.get(col_map.get(col, col), 0))
                    
                num_features.append(value)
            
            # Scale numerical features
            num_features = self.scaler.transform([num_features])[0]
            
            # Make prediction
            with torch.no_grad():
                cat_tensor = torch.tensor([cat_features], dtype=torch.long).to(DEVICE)
                num_tensor = torch.tensor([num_features], dtype=torch.float32).to(DEVICE)
                
                output = self.model(cat_tensor, num_tensor)
                prob = torch.sigmoid(output).item()
                
            predictions.append((
                driver_info.get('Driver', 'Unknown'),
                driver_info.get('Team', 'Unknown'), 
                prob
            ))
        
        # Sort by probability and return top 5
        predictions.sort(key=lambda x: x[2], reverse=True)
        
        result = f"Predictions for {grand_prix} {year}:\n\n"
        for i, (driver, team, prob) in enumerate(predictions[:5], 1):
            result += f"{i}. {driver} ({team}): {prob:.1%}\n"
            
        return result
    
    def create_interface(self):
        """Create Gradio interface"""
        
        with gr.Blocks(title="Zero-Shot F1 Predictor") as demo:
            gr.Markdown("# 🏎️ Zero-Shot F1 Winner Predictor")
            gr.Markdown("Predict F1 race winners using semantic understanding of drivers and teams!")
            
            with gr.Row():
                year_dropdown = gr.Dropdown(
                    choices=self.years,
                    value=self.years[-1] if self.years else None,
                    label="Year"
                )
                race_dropdown = gr.Dropdown(
                    choices=self.races,
                    value=self.races[0] if self.races else None,
                    label="Grand Prix"
                )
            
            predict_button = gr.Button("Predict Winners", variant="primary")
            
            output_text = gr.Textbox(
                label="Predictions",
                lines=8,
                interactive=False
            )
            
            predict_button.click(
                fn=self.predict_race,
                inputs=[year_dropdown, race_dropdown],
                outputs=output_text
            )
            
        return demo

def main():
    """Main execution function"""
    
    print("🏎️ Zero-Shot F1 Winner Prediction System")
    print("=" * 50)
    
    # Create data directory
    os.makedirs(DATA_DIR, exist_ok=True)
    
    try:
        # Load and process data
        winners, drivers, teams, fastest_laps = load_f1_data()
        drivers_with_features = create_driver_features(drivers, teams, fastest_laps)
        race_dataset = build_race_dataset(winners, drivers_with_features)
        
        # Prepare data for training
        (train_df, test_df, cat_cols, num_cols, 
         label_encoders, scaler, semantic_features, cat_dims) = prepare_data(race_dataset)
        
        # Create model
        model = ZeroShotF1Model(cat_dims, len(num_cols))
        model.to(DEVICE)
        model.set_semantic_features(semantic_features)
        
        # Create data loaders
        train_loader, test_loader = create_data_loaders(train_df, test_df, cat_cols, num_cols)
        
        # Train or load model
        if not os.path.exists(MODEL_PATH):
            train_model(model, train_loader, test_loader)
        else:
            print("Loading existing model...")
            checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
            model.load_state_dict(checkpoint['model_state_dict'])
        
        # Evaluate model
        evaluate_model(model, test_loader)
        
        # Create predictor and interface
        predictor = F1Predictor()
        predictor.load_model(MODEL_PATH, label_encoders, scaler, cat_cols, num_cols, semantic_features, cat_dims)
        predictor.setup_data(drivers_with_features, winners)
        
        # Launch interface
        demo = predictor.create_interface()
        
        print("\n🚀 Launching web interface...")
        demo.launch(share=False, server_name="0.0.0.0", server_port=7860)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()